# Asistente Bancario — RAG con Documentos del Banco

## GenAI Lifecycle: Adaptación (RAG) + Integración (PoC)

**Problema:** El modelo fine-tuneado sabe responder preguntas bancarias generales, pero no conoce los productos, tarifas ni políticas de un banco específico.

**Solución:** RAG (Retrieval-Augmented Generation) con documentos internos del "Banco Digital del Sur" (banco ficticio).

**PoC:** Chatbot que combina el conocimiento del fine-tuning + contexto específico del banco vía retrieval.

In [33]:
!!pip install -qU transformers accelerate peft bitsandbytes
!pip install -qU "langchain>=0.3,<1.0" "langchain-core>=0.3,<1.0" "langchain-community>=0.3,<1.0" langchain-huggingface faiss-cpu sentence-transformers


In [45]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from langchain.chains.retrieval_qa.base import RetrievalQA
from langchain.memory.buffer import ConversationBufferMemory
from langchain_community.vectorstores import FAISS
from langchain_core.vectorstores import VectorStoreRetriever
from huggingface_hub import notebook_login

import langchain
langchain.verbose = False

notebook_login()

In [41]:
HF_USERNAME = "testlegadoss"
adapter_id = f"{HF_USERNAME}/gemma3-4b-banking-assistant-es"
base_model_id = "unsloth/gemma-3-4b-it"  # Mismo modelo base usado en el fine-tuning

# Quantización 4-bit con compute en float16 (compatible con T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Cargar modelo base en 4-bit
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

# Aplicar el adapter LoRA del fine-tuning y poner en modo inferencia
model = PeftModel.from_pretrained(model, adapter_id)
model.training = False

# Tokenizer (el adapter guardó el tokenizer con el chat template de Gemma 3)
tokenizer = AutoTokenizer.from_pretrained(adapter_id)

text_generation_pipeline = transformers.pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.1,
    return_full_text=True,
    max_new_tokens=200,
)

banking_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Device set to use cuda:0


## 2. Knowledge base del banco

Creo documentos ficticios del "Banco Digital del Sur" con datos concretos: productos, comisiones, políticas. El punto es que este tipo de información el modelo no la puede saber solo con fine-tuning — necesita que se la pasemos como contexto. Por eso RAG.

In [42]:
bank_docs = [
    """Productos del Banco Digital del Sur: Ofrecemos Cuenta Ahorro (sin comisión de mantenimiento,
    rendimiento anual del 2.5%), Cuenta Corriente (comisión mensual de $5, incluye chequera),
    Tarjeta de Crédito Platinum (límite hasta $50,000, 3 cuotas sin interés en comercios adheridos),
    y Préstamo Personal (tasa fija del 15% anual, plazos de 12 a 60 meses).""",

    """Comisiones y Tarifas: Transferencias entre cuentas del banco: gratis. Transferencias a otros
    bancos: $2 por operación. Retiro en cajeros propios: gratis. Retiro en cajeros de otra red: $3.
    Mantenimiento Cuenta Corriente: $5/mes. Emisión de tarjeta de débito: gratis. Reposición por
    pérdida: $10.""",

    """Banca Digital y App Móvil: La app 'BDS Móvil' permite abrir cuentas en 5 minutos con DNI
    digital, realizar transferencias, pagar servicios, invertir en plazos fijos, y contactar soporte
    por chat 24/7. Disponible en iOS y Android. Límite de transferencia diario por app: $100,000.""",

    """Política de Reclamos: Los reclamos se procesan en un máximo de 10 días hábiles. Para cobros
    no reconocidos en tarjeta de crédito, el banco realiza un reembolso provisorio dentro de las 48hs
    mientras investiga. Canales: app, web, teléfono (0800-555-1234), o sucursal.""",

    """Préstamos y Financiación: Préstamo Personal: desde $10,000 hasta $500,000, tasa fija 15% anual,
    plazos 12-60 meses, aprobación en 24hs. Préstamo Hipotecario: hasta 80% del valor de la propiedad,
    tasa variable UVA + 3.5%, plazos hasta 30 años. Requisitos: antigüedad laboral mínima 6 meses,
    recibo de sueldo, DNI.""",
]

In [43]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
)

vector_db = FAISS.from_texts(bank_docs, embeddings)
retriever = VectorStoreRetriever(vectorstore=vector_db)

## 4. Armar el chain RAG

Uso el template de chat de Gemma 3 directamente en el prompt para que el modelo reciba la entrada en el formato que conoce. Le paso el contexto recuperado por FAISS y un historial de conversación para que pueda mantener el hilo si le hacen preguntas de seguimiento.

In [46]:
system_message = "Eres un asistente virtual del Banco Digital del Sur. Responde siempre en español, de forma clara y profesional."

template = """<bos><start_of_turn>user
{system_message}

Contexto del banco:
{context}

Historial: {history}

Pregunta del cliente: {question}
<end_of_turn>
<start_of_turn>model
"""

prompt = PromptTemplate(
    template=template.replace("{system_message}", system_message),
    input_variables=["history", "context", "question"]
)

qa = RetrievalQA.from_chain_type(
    llm=banking_llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={
        "verbose": False,
        "prompt": prompt,
        "memory": ConversationBufferMemory(
            memory_key="history",
            input_key="question"
        ),
    }
)

## 5. Demo del chatbot

Pruebo con preguntas que requieren datos específicos del banco (tarifas, tasas, productos). Si el RAG funciona bien, las respuestas deberían incluir los números concretos de los documentos. Sin RAG, el modelo solo podría dar respuestas genéricas.

In [47]:
demo_questions = [
    "¿Qué tipos de cuenta ofrecen?",
    "¿Cuál es la tasa del préstamo personal?",
    "¿Cómo abro una cuenta desde la app?",
    "Quiero hacer un reclamo por un cobro no reconocido",
    "¿Cuánto cuesta transferir a otro banco?",
]

for q in demo_questions:
    print(f"\n{'='*60}")
    print(f"Cliente: {q}")
    response = qa.run(q)
    print(f"Asistente: {response}")


Cliente: ¿Qué tipos de cuenta ofrecen?


/tmp/ipython-input-4140058677.py:12: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response = qa.run(q)


Asistente: <bos><start_of_turn>user
Eres un asistente virtual del Banco Digital del Sur. Responde siempre en español, de forma clara y profesional.

Contexto del banco:
Comisiones y Tarifas: Transferencias entre cuentas del banco: gratis. Transferencias a otros
    bancos: $2 por operación. Retiro en cajeros propios: gratis. Retiro en cajeros de otra red: $3.
    Mantenimiento Cuenta Corriente: $5/mes. Emisión de tarjeta de débito: gratis. Reposición por
    pérdida: $10.

Productos del Banco Digital del Sur: Ofrecemos Cuenta Ahorro (sin comisión de mantenimiento,
    rendimiento anual del 2.5%), Cuenta Corriente (comisión mensual de $5, incluye chequera),
    Tarjeta de Crédito Platinum (límite hasta $50,000, 3 cuotas sin interés en comercios adheridos),
    y Préstamo Personal (tasa fija del 15% anual, plazos de 12 a 60 meses).

Política de Reclamos: Los reclamos se procesan en un máximo de 10 días hábiles. Para cobros
    no reconocidos en tarjeta de crédito, el banco realiza un ree

### Conclusiones — Fase Adaptación (RAG) + Integración

### ¿El RAG mejora las respuestas?

Sí, el RAG mejora las respuestas en un aspecto clave: le da al modelo acceso a datos concretos del banco que antes no tenía. En el Notebook 3 vimos que el modelo fine-tuneado podía inventar cifras o datos porque no tenía de dónde sacarlos. Ahora, con el RAG, cuando el cliente pregunta "¿Cuánto cuesta transferir a otro banco?", el sistema busca en los documentos del banco y le pasa al modelo el dato real ($2 por operación). Lo mismo con la tasa del préstamo (15% anual) o los plazos de reclamo (10 días hábiles, reembolso en 48hs). Dicho esto, la mejora tiene límites: el RAG aporta los datos correctos, pero la forma en que el modelo arma la respuesta sigue dependiendo de sus propias capacidades. En las pruebas noté que algunas respuestas se cortaban a la mitad por el límite de 200 tokens, otras mezclaban español con inglés en algunas frases, y en un caso el modelo no respondió directamente la pregunta (le preguntaron por tipos de cuenta y empezó hablando de préstamos). El RAG no corrige esos problemas de generación, solo garantiza que la información de base sea la correcta.

### ¿Qué tipo de preguntas se benefician más?

Las preguntas que más se benefician son las que necesitan datos específicos del banco: precios, tasas, montos y políticas. Por ejemplo, "¿Cuánto cuesta transferir a otro banco?" o "Quiero hacer un reclamo por un cobro no reconocido" son preguntas donde la respuesta correcta depende de información que solo existe en los documentos internos (el costo de $2, el reembolso provisorio en 48hs, el teléfono 0800-555-1234). Sin el RAG, el modelo tendría que inventar esos datos. En cambio, preguntas más generales como "¿Cómo abro una cuenta desde la app?" dependen más de que el modelo pueda generar una explicación paso a paso coherente, y ahí el RAG ayuda menos — puede pasarle el dato de que la app permite abrir cuentas en 5 minutos con DNI digital, pero armar una respuesta clara es tarea del modelo.

### PoC completada

Con este notebook queda terminada la Prueba de Concepto del Asistente Bancario. El proyecto completo siguió las cuatro fases del GenAI Project Lifecycle que vimos en el bootcamp: primero construimos el dataset traduciendo datos bancarios de inglés a español (Notebook 1), después adaptamos el modelo Gemma 3 4B con fine-tuning usando QLoRA (Notebook 2), luego comparamos las respuestas del modelo base vs el fine-tuneado (Notebook 3), y finalmente en este Notebook 4 agregamos RAG con documentos del banco e integramos todo en un chatbot funcional. El resultado es un asistente que combina lo que aprendió en el fine-tuning (tono profesional, comportarse como agente bancario) con los datos reales del Banco Digital del Sur que le llegan por RAG mediante FAISS. Además, la memoria conversacional permite que el asistente recuerde las preguntas anteriores dentro de una misma sesión.

### Limitaciones y próximos pasos

Durante las pruebas identifiqué varias limitaciones. Las respuestas tienden a cortarse a mitad de frase porque el límite de generación está en 200 tokens, lo cual se podría mejorar aumentando ese valor. En algunas respuestas el modelo mezcla español con inglés (frases como "through the rest of the proceso" o "keep an eye on your statement"), lo que muestra que el fine-tuning no alcanzó para que el modelo responda 100% en español. También noté que a veces el modelo no va directo al punto: cuando le preguntan por tipos de cuenta, arranca hablando de préstamos en vez de listar las cuentas disponibles. Otro punto a mejorar es que el modelo de embeddings que usé (`all-MiniLM-L6-v2`) está entrenado en inglés, y los documentos del banco están en español; un modelo multilingüe probablemente recuperaría documentos más relevantes. Por último, la evaluación fue manual sobre solo 5 preguntas; para un proyecto más completo habría que usar métricas automáticas. Como mejoras futuras se podrían explorar: agregar una interfaz web, ampliar la cantidad de documentos del banco, y probar con un modelo de embeddings multilingüe para mejorar la búsqueda.